# Average the maps

Just a small programm, averaging the most constant, and best maps, for later use 
___________
Erstellt am 20.07.2026 von Gregor Bock

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import External_functions as fkt

## Extract Data

In [ ]:
# Specify folder path for .npz and points folder (.npz file should contain L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
folder_path_1 = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Finale_Karten\2026-07-16_16-59-44_I1 P I2\maps\map\map\points"
folder_path_2 = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Finale_Karten\2026-07-16_21-21-00_I1 P I2\maps\map\map\points"
folder_path_3 = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Finale_Karten\2026-07-17_01-49-13_I1 P I2\maps\map\map\points"

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction -> Old feature, no more meaning. Kept if useful later
L_y = 0.800                                         # length of the mapped volume in y-direction -> Old feature, no more meaning. Kept if useful later
L_z = 0.400                                         # length of the mapped volume in z-direction -> Old feature, no more meaning. Kept if useful later
step_size = 0.200                                   # size of the grid steps                     -> Old feature, no more meaning. Kept if useful later

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

# Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
coord_exp, B_map_1, B_map_2 = fkt.load_data_from_folder(folder_path_1, folder_path_2, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
coord_exp_2, B_map_3, _ = fkt.load_data_from_folder(folder_path_1, folder_path_3, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)

if np.all(coord_exp == coord_exp_2):
    print(f'All good :). Coordinates of all maps are the same.')
else:
    print(f'FUCK: Coordinates are not the same for the three maps :(')

## Plot Data

In [ ]:
x = coord_exp[:,0]
y = coord_exp[:,1]
z = coord_exp[:,2]
Bnorm1 = np.linalg.norm(B_map_1, axis=1)
Bnorm2 = np.linalg.norm(B_map_2, axis=1)
Bnorm3 = np.linalg.norm(B_map_3, axis=1)

fig1 = plt.figure()
ax1 = fig1.add_subplot(111, projection='3d')

scat1 = ax1.scatter(x, y, z, c=Bnorm1, s=100, cmap='viridis', alpha = 0.8)

ax1.set_xlabel('x [m]')
ax1.set_ylabel('y [m]')
ax1.set_zlabel('z [m]')
ax1.set_title('Magnetic Field Strength at Target Points')
cbar1 = fig1.colorbar(scat1, ax=ax1, label='|B| (T)')
cbar1.ax.set_position([0.85, 0.15, 0.03, 0.7])


fig2 = plt.figure()
ax2 = fig2.add_subplot(111, projection='3d')

scat2 = ax2.scatter(x, y, z, c=Bnorm2, s=100, cmap='viridis', alpha = 0.8)

ax2.set_xlabel('x [m]')
ax2.set_ylabel('y [m]')
ax2.set_zlabel('z [m]')
ax2.set_title('Magnetic Field Strength at Target Points')
cbar2 = fig2.colorbar(scat2, ax=ax2, label='|B| (T)')
cbar2.ax.set_position([0.85, 0.15, 0.03, 0.7])


fig3 = plt.figure()
ax3 = fig3.add_subplot(111, projection='3d')

scat3 = ax3.scatter(x, y, z, c=Bnorm3, s=100, cmap='viridis', alpha = 0.8)

ax3.set_xlabel('x [m]')
ax3.set_ylabel('y [m]')
ax3.set_zlabel('z [m]')
ax3.set_title('Magnetic Field Strength at Target Points')
cbar3 = fig3.colorbar(scat3, ax=ax3, label='|B| (T)')
cbar3.ax.set_position([0.85, 0.15, 0.03, 0.7])

plt.show()

## Average

In [ ]:
B_stack = np.stack([B_map_1, B_map_2, B_map_3], axis=0)
B_average = np.average(B_stack, axis=0)

B_norm_av = np.linalg.norm(B_average, axis=1)

fig_av = plt.figure()
ax_av = fig_av.add_subplot(111, projection='3d')

scat_av = ax_av.scatter(x, y, z, c=B_norm_av, s=100, cmap='viridis', alpha=0.8)

ax_av.set_xlabel('x [m]')
ax_av.set_ylabel('y [m]')
ax_av.set_zlabel('z [m]')
ax_av.set_title('Magnetic Field Strength at Target Points')
cbar_av = fig_av.colorbar(scat_av, ax=ax_av, label='|B| (T)')
cbar_av.ax.set_position([0.85, 0.15, 0.03, 0.7])

## Write averaged map

In [ ]:
target_path = Path("D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Finale_Karten\\Averaged_B_field")

# Create directory if it doesn't exist
target_path.mkdir(parents=True, exist_ok=True)

# Convert units: meters to millimeters, Tesla to picotesla
B_in_pT = B_average * 10**(12)

# Save each point as a separate .npz file
num_points = coord_exp.shape[0]
for i in range(num_points):
    # Extract coordinates for this point (convert m to mm)
    x_mm = coord_exp[i, 0] * 1e3
    y_mm = coord_exp[i, 1] * 1e3
    z_mm = coord_exp[i, 2] * 1e3
    
    # Extract B-field values for this point (in pT)
    mean_Bx_pT = B_in_pT[i, 0]
    mean_By_pT = B_in_pT[i, 1]
    mean_Bz_pT = B_in_pT[i, 2]
    
    # Create filename (pad with zeros for consistency)
    filename = target_path / f"point_{i+1:03d}.npz"
    
    # Save as dictionary in .npz file
    np.savez(filename, 
             x_mm=x_mm, 
             y_mm=y_mm, 
             z_mm=z_mm, 
             mean_Bx_pT=mean_Bx_pT, 
             mean_By_pT=mean_By_pT, 
             mean_Bz_pT=mean_Bz_pT)

print(f"Saved {num_points} points to {target_path}")